## LunaNet Demonstration

In [ ]:
import pylupnt as pnt
import numpy as np

In [ ]:
plot_case = "lunanet_demo"
plot_case = "lcrns"

if plot_case == "lunanet_demo":
    sma = (
        np.array([9748.14, 3870.00, 11999.2626, 12027.7960, 11993.3508]) * 1000
    )  # in meters
    ecc = np.array([0.7, 0.0001, 0.655, 0.641, 0.721])
    inc = np.array([48.04, 104.428, 32.22, 31.33, 79.07]) * np.pi / 180  # in radians
    RAAN = (
        np.array([89.49, 53.563, -162.33, -164.02, -42.86]) * np.pi / 180
    )  # in radians
    w = np.array([123.60, 90.0, 75.96, 76.14, 68.18]) * np.pi / 180  # in radians
    f = np.array([90.0, -5.0, -147.49, 151.30, -119.79]) * np.pi / 180  # in radians
    M = pnt.true_to_mean_anomaly(f, ecc)
    nsat = len(sma)
    print(f"Number of satellites: {nsat}")

    t0_tai = pnt.convert_time(
        pnt.gregorian_to_time(2027, 1, 1, 0, 0, 0), pnt.TDB, pnt.TAI
    )
elif plot_case == "lcrns":
    svoe = np.zeros((5, 6))
    svoe[0] = np.array(
        [11315.936501, 0.691982, 59.373229, 321.019197, 92.494031, 0.000000]
    )
    svoe[1] = np.array(
        [11317.948675, 0.691982, 58.951732, 320.997768, 92.505016, 180.000000]
    )
    svoe[2] = np.array(
        [11305.413654, 0.691982, 52.733096, 81.148790, 92.062891, 140.049207]
    )
    svoe[3] = np.array(
        [11326.302154, 0.691982, 52.513419, 81.138818, 92.068945, 195.992393]
    )
    svoe[4] = np.array(
        [11307.882863, 0.691982, 56.310396, 204.889626, 85.444071, 164.007607]
    )
    svoe[:, 0] = svoe[:, 0] * 1000  # sma
    svoe[:, 2:] = np.deg2rad(svoe[:, 2:])
    nsat = svoe.shape[0]
    sma = svoe[:, 0]
    ecc = svoe[:, 1]
    inc = svoe[:, 2]
    RAAN = svoe[:, 3]
    w = svoe[:, 4]
    M = svoe[:, 5]

    t0_tai = pnt.convert_time(
        pnt.gregorian_to_time(2027, 3, 1, 0, 0, 0), pnt.UTC, pnt.TAI
    )


tspan = np.linspace(0, 48 * 3600, 48 * 60 + 1)  # 0 to 48 hours, 500 points
t_tai = t0_tai + tspan

# Dynamics ----------------------------------------------------------------
pnt.set_lupnt_epoch(0)
dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(
    pnt.IntegratorParams(max_iter=100, abstol=1e-12, reltol=1e-12)
)
dyn.add_body(pnt.Body.Moon(2, 2))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_srp_coeff(CR=1.8, area=1.0, mass=850.0)
dyn.set_time_step(60.0)  # 60 seconds

x_prop = np.zeros((nsat, len(tspan), 6))
for i in range(nsat):
    oe = np.array([sma[i], ecc[i], inc[i], RAAN[i], w[i], M[i]])
    rv0 = pnt.classical_to_cart(oe, pnt.GM_MOON)
    print(f"Propagating satellite {i+1}/{nsat}... rv0 = {rv0}")
    x_prop[i] = dyn.propagate(rv0, t_tai)

In [ ]:
import plotly.express as px


def plot_orbits(
    fig: go.Figure,
    rv: np.ndarray,
    color,
    t: int = None,
    marker_size: float = 4,
    scale: float = 6,
    linestyle: str = "solid",
    width: float = 3,
) -> go.Figure:
    rv = rv / 10**scale

    if rv.ndim == 2:
        rv = rv[np.newaxis, :, :]

    N_sat = rv.shape[0]
    for i in range(N_sat):
        fig.add_scatter3d(
            **dict(x=rv[i, :, 0], y=rv[i, :, 1], z=rv[i, :, 2]),
            mode="lines",
            line=dict(
                color=color[i % len(color)] if type(color) == list else color,
                width=width,
                dash=linestyle,
            ),
            name=f"plot_orbits_{i}",
            showlegend=False,
        )
    if t is not None:
        fig.add_scatter3d(
            **dict(x=rv[:, t, 0], y=rv[:, t, 1], z=rv[:, t, 2]),
            mode="markers",
            marker=dict(
                color=color, size=marker_size, line=dict(color=color, width=0.5)
            ),
            name="plot_orbits_markers",
            showlegend=False,
        )

    # Dummy trace
    c = 220
    axis_dict = dict(
        linecolor=f"rgb({c},{c},{c})",
        gridcolor=f"rgb({c},{c},{c})",
        linewidth=1.5,
        gridwidth=1.5,
        showline=False,
        mirror=True,
    )
    fig.update_layout(
        scene=dict(
            xaxis=dict(title="X [10<sup>3</sup> km]", **axis_dict),
            yaxis=dict(title="Y [10<sup>3</sup> km]", **axis_dict),
            zaxis=dict(title="Z [10<sup>3</sup> km]", **axis_dict),
            # xaxis=dict(title="X [10<sup>3</sup> km]", **axis_dict),
            # yaxis=dict(title="Y [10<sup>3</sup> km]", **axis_dict),
            # zaxis=dict(title="Z [10<sup>3</sup> km]", **axis_dict),
        ),
        font=dict(size=12, family="serif"),
        margin=dict(l=10, r=10, t=10, b=10),
        height=600,
        width=600,
        legend=dict(
            x=0.9,
            y=0.85,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255, 255, 255, 0.5)",
        ),
    )
    fig.update_layout(scene_aspectmode="data")
    xticks = fig.layout.scene.xaxis.tickvals
    yticks = fig.layout.scene.yaxis.tickvals
    zticks = fig.layout.scene.zaxis.tickvals
    if xticks is not None and yticks is not None and zticks is not None:
        fig.update_layout(
            scene=dict(
                xaxis_ticktext=[f"{x / 1e3:.0f}" for x in xticks],
                yaxis_ticktext=[f"{y / 1e3:.0f}" for y in yticks],
                zaxis_ticktext=[f"{z / 1e3:.0f}" for z in zticks],
            )
        )

    return fig

In [ ]:
import plotly.graph_objects as go
import os

fig = go.Figure()

if plot_case == "lunanet_demo":
    colors = ["yellow", "red", "blue", "blue", "blue"]
elif plot_case == "lcrns":
    colors = ["red", "red", "blue", "blue", "yellow"]

pnt.plot.plot_body(
    fig,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)

if plot_case == "lunanet_demo":
    pnt.plot.set_view(fig, 120, 20, 1.4)
    tdix = 2100
else:
    pnt.plot.set_view(fig, 120, 20, 1.8)
    tidx = 0

fig.update_layout(showlegend=False, width=400, height=400)

for i in range(nsat):
    # line to lunar south pole
    xlines = np.array(
        [
            [x_prop[i, tidx, 0], x_prop[i, tidx, 1], x_prop[i, tidx, 2]],
            [0, 0, -1737.4e3],
        ]
    )
    plot_orbits(fig, x_prop[i], color=colors[i], linestyle="solid")  # [N, t, 3]
    pnt.plot.scatter(fig, x_prop[i, tidx, :3], color=colors[i])
    if plot_case == "lunanet_demo":
        plot_orbits(
            fig, xlines, color=colors[i], linestyle="dash"
        )  # line to south pole
    elif plot_case == "lcrns" and i != 0:
        plot_orbits(
            fig, xlines, color=colors[i], linestyle="dash"
        )  # line to south pole

# make background black
fig.update_layout(paper_bgcolor="black", plot_bgcolor="black")
# delete axis
fig.update_layout(
    scene=dict(
        xaxis=dict(
            showbackground=False, visible=False, showticklabels=False, showgrid=False
        ),
        yaxis=dict(
            showbackground=False, visible=False, showticklabels=False, showgrid=False
        ),
        zaxis=dict(
            showbackground=False, visible=False, showticklabels=False, showgrid=False
        ),
    )
)
# delete labels
fig.update_layout(scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""))

# save the figure as pdf
basepath = pnt.get_output_dir()
orbdir = os.path.join(basepath, "iono_delay", "orbits")
if not os.path.exists(orbdir + "/figures"):
    os.makedirs(orbdir + "/figures")

if plot_case == "lunanet_demo":
    fig.write_image(orbdir + "/figures/demo_mission_orbits.pdf")
elif plot_case == "lcrns":
    fig.write_image(orbdir + "/figures/lcrns_mission_orbits.pdf")
fig.show()